In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, json, joblib, warnings

from sklearn.model_selection import cross_validate, train_test_split, learning_curve
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore', category=UserWarning)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
root = '/content/drive/MyDrive/Portnet Imputation Prediction'
pipeline = root + "/src"

raw_data = pd.read_csv(f'{root}/data.dsv', sep=';')
if pipeline not in sys.path: sys.path.append(pipeline)

from data_pipeline import portnet_pipeline

In [10]:
from huggingface_hub import notebook_login, hf_hub_download, HfApi

notebook_login()

In [4]:
repo_id = "Meliodas-10/portnet-model"

model_file = hf_hub_download(repo_id=repo_id, filename="portnet_model.pkl")
params_file = hf_hub_download(repo_id=repo_id, filename="best_params.json")

full_model = joblib.load(model_file)
with open(params_file, "r") as f: best_params = json.load(f)

print("Modèle et paramètres rechargé depuis Hugging Face !")

Modèle et paramètres rechargé depuis Hugging Face !


In [ ]:
df = raw_data[raw_data['QTE_IMPUTE'] > 0].copy()
df = df[df['DEVISE'].isin(['EUR', 'USD'])]

X = df.drop(columns=['QTE_IMPUTE'])
y = df['QTE_IMPUTE']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 1- Learning_curve

In [ ]:
print("--- Génération de la Learning Curve ---")

# On utilise un échantillon pour calculer la courbe plus rapidement
X_sample = X_train.sample(n=500000, random_state=42)
y_sample = y_train.loc[X_sample.index]

train_sizes, train_scores, val_scores = learning_curve(
    full_model, 
    X_sample, 
    y_sample, 
    cv=3, 
    scoring='r2',
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1
)

# Calcul des moyennes et écarts-types
train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

# Tracé du graphique
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, label="Score d'entraînement", marker='o', color='blue')
plt.plot(train_sizes, val_mean, label="Score de validation", marker='s', color='orange')

plt.title("Courbe d'apprentissage (Learning Curve) - Modèle Portnet")
plt.xlabel("Taille du dataset d'entraînement")
plt.ylabel("Score R²")
plt.legend(loc="best")
plt.grid(True)
plt.show()

# 2- Cross-Validation

In [ ]:
model_val = TransformedTargetRegressor(
    regressor=portnet_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# Application propre des paramètres chargés depuis Hugging Face
cleaned_params = {}
for k, v in best_params.items():
    if not k.startswith('algorithme__'):
        cleaned_params[f'regressor__algorithme__{k.replace("regressor__", "")}'] = v
    else:
        cleaned_params[f'regressor__{k}'] = v

model_val.set_params(**cleaned_params)
print("Modèle de validation configuré avec succès !")

In [ ]:
print("--- Lancement de la Validation Croisée (CV = 5) ---")

cv_results = cross_validate(
    model_val,
    X_train, 
    y_train,
    cv=5,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

print("Validation croisée terminée avec succès !")

In [ ]:
folds_r2 = cv_results['test_r2']
folds_mae = -cv_results['test_neg_mean_absolute_error'] if 'test_neg_mean_absolute_error' in cv_results else -cv_results['test_mae']
folds_rmse = -cv_results['test_neg_root_mean_squared_error'] if 'test_neg_root_mean_squared_error' in cv_results else -cv_results['test_rmse']

print("\n" + "="*50)
print("RAPPORT DE VALIDATION DIAGNOSTIQUE (CV = 5)")
print("="*50)
for i in range(5):
    print(f"Pli {i+1} -> R²: {folds_r2[i]:.4f} | MAE: {folds_mae[i]:,.2f} | RMSE: {folds_rmse[i]:,.2f}")

print("-"*50)
print(f"Moyenne R²   : {folds_r2.mean():.4f} (± {folds_r2.std():.4f})")
print(f"Moyenne MAE  : {folds_mae.mean():,.2f} (± {folds_mae.std():,.2f})")
print(f"Moyenne RMSE : {folds_rmse.mean():,.2f} (± {folds_rmse.std():,.2f})")
print("="*50)

# 3- Metrics

In [ ]:
y_pred = full_model.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
nrmse_base = rmse_base / np.mean(y_test)
r2_base = r2_score(y_test, y_pred)

print("\n--- Final Score ---")
print(f"MAE                     : {mae_base:.2f}")
print(f"RMSE                    : {rmse_base:.2f}")
print(f"Moyenne de y_test       : {np.mean(y_test):.2f}")
print(f"NRMSE(goal: [0.2;0.4])  : {nrmse_base:.4f} ({nrmse_base * 100:.2f}%)")
print(f"R²                      : {r2_base:.4f}")

[_drop_unnecessary_columns] exécuté en 0.080s | Lignes restantes: 922972
[_clean_quantite_domicile] exécuté en 0.005s | Lignes restantes: 922972
[_set_datatypes] exécuté en 77.934s | Lignes restantes: 922972
[_optimize_memory] exécuté en 0.068s | Lignes restantes: 922972
[_delai_extracting] exécuté en 0.068s | Lignes restantes: 922972
[_strategic_grouping] exécuté en 0.006s | Lignes restantes: 922972
[_temporal_engineering] exécuté en 0.075s | Lignes restantes: 922972
[_flags_creation] exécuté en 0.003s | Lignes restantes: 922972
[_unify_currency_to_eur] exécuté en 0.264s | Lignes restantes: 922972
[_logarithmic_transform] exécuté en 0.010s | Lignes restantes: 922972
[_impute_and_scale] exécuté en 0.053s | Lignes restantes: 922972
[_drop_useless_text] exécuté en 0.025s | Lignes restantes: 922972
[_target_encode] exécuté en 0.253s | Lignes restantes: 922972
--- Final Score ---
MAE  : 15941.86
RMSE : 498138.42
R²   : 0.9535


In [ ]:
metrics = {
    "r2_score": float(r2_base),
    "mae": float(mae_base),
    "rmse": float(rmse_base),
    "nrmse": float(nrmse_base),
    "dataset_rows": len(X_train)
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

api = HfApi()
api.upload_file(
    path_or_fileobj="metrics.json",
    path_in_repo="metrics.json",
    repo_id=repo_id,
    repo_type="model"
)
print("Métriques de performance versionnées avec succès sur Hugging Face !")

Métriques de performance versionnées avec succès sur Hugging Face !
